## Table of Contents

1.  [**Introduction**](#introduction)
    *   1.1. [Project Overview](#project-overview)
    *   1.2. [About the Dataset](#about-the-dataset)
    *   1.3. [Objectives](#objectives)

2.  [**Setup and Initial Configuration**](#setup)
    *   2.1. [Importing Libraries](#importing-libraries)
    *   2.2. [Loading the Dataset](#loading-the-dataset)

3.  [**Data Exploration and Understanding**](#data-exploration)
    *   3.1. [Initial Data Inspection (`.head()`, `.tail()`)](#initial-inspection)
    *   3.2. [Data Structure and Types (`.info()`)](#data-structure)
    *   3.3. [Summary Statistics (`.describe()`)](#summary-statistics)
    *   3.4. [Checking for Missing Values](#missing-values)

4.  [**Data Preprocessing and Cleaning**](#data-preprocessing)
    *   4.1. [Handling Timestamps](#handling-timestamps)
        *   4.1.1. [Converting String Columns to Datetime Objects](#converting-datetimes)
    *   4.2. [Filtering for the Analysis Period (Q2 2016)](#filtering-data)
    *   4.3. [Handling Missing or Inconsistent Data](#handling-inconsistent-data)

5.  [**Feature Engineering: Time to Resolution**](#feature-engineering)
    *   5.1. [Calculating the Time Difference](#calculating-time-difference)
    *   5.2. [Converting Time Delta to a Numerical Unit (e.g., Hours, Days)](#converting-time-delta)
    *   5.3. [Inspecting the New Feature](#inspecting-new-feature)

6.  [**Exploratory Data Analysis (EDA)**](#eda)
    *   6.1. [Analyzing the Distribution of Resolution Times](#distribution-analysis)
        *   6.1.1. [Histogram and Kernel Density Estimate](#histogram)
        *   6.1.2. [Box Plot to Identify Outliers](#box-plot)
    *   6.2. [Resolution Time by Ticket Priority](#priority-analysis)
    *   6.3. [Resolution Time by Assignment Group](#group-analysis)
    *   6.4. [Resolution Time by Ticket Category](#category-analysis)
    *   6.5. [Trends Over Time within Q2 2016](#trends)

7.  [**Insights and Findings**](#insights)
    *   7.1. [Summary of Key Metrics](#key-metrics)
    *   7.2. [Identifying Bottlenecks and Efficiencies](#bottlenecks)
    *   7.3. [Answering Key Business Questions](#business-questions)

8.  [**Conclusion**](#conclusion)
    *   8.1. [Summary of Analysis](#summary-of-analysis)
    *   8.2. [Recommendations and Next Steps](#recommendations)

## Read the data

In [1]:
from importlib.resources import files
import pandas as pd
fp = str(files("kmds.examples").joinpath("q2_2016_ticket_resolution_data.csv"))
df = pd.read_csv(fp)


## Verify Quality

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3807 entries, 0 to 3806
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   number            3807 non-null   str  
 1   sys_created_at    3807 non-null   str  
 2   closed_at         3807 non-null   str  
 3   assignment_group  3380 non-null   str  
dtypes: str(4)
memory usage: 324.2 KB


## Define the data types

In [3]:

fpdtypes = str(files("kmds.examples").joinpath("ticket_resolution_dtypes.csv"))
dtypes_df = pd.read_csv(fpdtypes)
dtypes_dict = {row["attribute"]: row["type"] for index, row in dtypes_df.iterrows()}
df = df.astype(dtypes_dict)
df = df.reset_index()

## Feature Engineering
Define the resolution time attribute as shown below

In [4]:
df["resolution_time"] = df["closed_at"] - df["sys_created_at"]
df["resolution_time"] = df["resolution_time"].apply(lambda x: x.total_seconds()/3600)

In [5]:
df["resolution_time"]

0       1546.616667
1       2285.050000
2       1544.566667
3       1545.716667
4       2282.716667
           ...     
3802     126.283333
3803    3649.166667
3804    2929.383333
3805    3648.233333
3806    1475.616667
Name: resolution_time, Length: 3807, dtype: float64

## Write the representation to disk

## Log Data Representation Observations to KMDS Knowledge Base

In [6]:
from kmds.tagging.tag_types import DataRepresentationTags
from owlready2 import *
from kmds.utils.load_utils import *
#from utils.path_utils import *
KNOWLEDGE_BASE = str(files("kmds.examples").joinpath("example_analytics_kb_app_workflow.xml"))

* Owlready2 * Warning: optimized Cython parser module 'owlready2_optimized' is not available, defaulting to slower Python implementation


In [7]:
onto2 = load_kb(KNOWLEDGE_BASE)

In [8]:
with onto2:
    insts = Workflow.instances()
the_workflow_instance = insts[0]

In [9]:
insts

[example_analytics_kb_app_workflow.xml.ITSM modelling]

In [10]:
from kmds.ontology.intent_types import IntentType
dr_obs_list = []
observation_count = 1

dr1 = DataRepresentationObservation(namespace=onto2)
dr1.finding = "The resolution time attribute is defined. It is calculated as the time difference between closing and creation\
times of the ticket."
dr1.finding_sequence = observation_count
dr1.data_representation_observation_type = DataRepresentationTags.FEATURE_ENGG_OBSERVATION.value
dr1.intent = IntentType.FEATURE_ASSESSMENT.value
dr_obs_list.append(dr1)
the_workflow_instance.has_data_representation_observations = dr_obs_list

onto2.save(file=KNOWLEDGE_BASE, format="rdfxml")